# Exploring Distributions

In the previous notebook we calculated medians, standard deviations, and IQRs. Those numbers told us something useful, but they also hid a lot. Two datasets can share the same median and the same IQR yet look completely different when you plot them.

The explainer we just read introduced the idea that the *shape* of data determines which statistics are appropriate and which will mislead. A right-skewed distribution needs the median. A bimodal distribution needs to be split before you summarise it at all. We cannot know any of this from summary statistics alone.

This notebook puts that idea into practice. We will build histograms and box plots of the ED data, see the skewness we measured numerically in the last notebook, and compare distributions across patient groups.

By the end of this notebook we will be able to:

- Build **histograms** with `plt.hist()` and `sns.histplot()` and choose an appropriate bin count
- Build **box plots** with `plt.boxplot()`, `df.boxplot()`, and `sns.boxplot()`
- Identify **skewness** visually and explain what it means for our data
- Compare distributions across groups using side-by-side plots

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
arrivals = pd.read_csv("../data/ed_arrivals.csv", parse_dates=["arrival_time"])
waits = pd.read_csv("../data/ed_wait_times.csv")
df = arrivals.merge(waits, on="patient_id")
df.head()

---

## 1. Histograms: seeing the shape of data

A **histogram** divides a numeric variable into equal-width bins and counts how many values fall into each bin. It is the visual equivalent of the frequency distribution table from the explainer. The shape it reveals tells us whether the data is symmetric, skewed, or multimodal.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["wait_to_treatment_min"].dropna(), bins=40, edgecolor="white")
plt.xlabel("Wait to treatment (minutes)")
plt.ylabel("Number of patients")
plt.title("Distribution of wait to treatment")
plt.show()

There it is: the long right tail. Most patients are treated within a reasonable time, but a minority wait much longer. This is **right skew** (positive skew), and it is exactly the shape the explainer described for A&E wait times. The mean gets pulled toward that tail, which is why the median was the better summary statistic in the previous notebook.

### Choosing the number of bins

The number of bins affects what we can see. Too few bins hide detail. Too many create noise. Try different values and watch what happens.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, bins in zip(axes, [10, 40, 100]):
    ax.hist(df["wait_to_treatment_min"].dropna(), bins=bins, edgecolor="white")
    ax.set_title(f"bins={bins}")
    ax.set_xlabel("Wait to treatment (min)")

plt.tight_layout()
plt.show()

With 10 bins, the shape is too coarse to see much. With 100, individual bars start jumping around. Somewhere between 30 and 50 is usually a good starting point for a dataset of this size. There is no single correct answer; the goal is to reveal the shape without inventing patterns that are not there.

### Using seaborn's `histplot`

Seaborn wraps matplotlib and can overlay a **kernel density estimate** (KDE) line. The KDE is a smooth curve that estimates the underlying distribution shape, making it easier to see the overall pattern without being distracted by individual bars.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["wait_to_treatment_min"].dropna(), bins=40, kde=True)
plt.xlabel("Wait to treatment (minutes)")
plt.title("Distribution of wait to treatment (with KDE)")
plt.show()

---

## 2. Histograms for other variables

One histogram is good. Several are better. Let's check the distribution of wait to triage and total time in ED to see whether the right-skew pattern holds across different measures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["wait_to_triage_min"].dropna(), bins=40, edgecolor="white", color="steelblue")
axes[0].set_xlabel("Wait to triage (minutes)")
axes[0].set_ylabel("Number of patients")
axes[0].set_title("Wait to triage")

axes[1].hist(df["total_time_in_ed_min"].dropna(), bins=40, edgecolor="white", color="coral")
axes[1].set_xlabel("Total time in ED (minutes)")
axes[1].set_ylabel("Number of patients")
axes[1].set_title("Total time in ED")

plt.tight_layout()
plt.show()

Both are right-skewed. This is typical for time-based measurements: there is a natural lower bound (you cannot wait less than zero minutes) but no hard upper bound, so the tail stretches to the right. Knowing this, we can be confident that medians are the right choice for all three wait-time columns.

---

## 3. Box plots: summarising a distribution in five numbers

A **box plot** shows the **five-number summary**: minimum, Q1, median, Q3, maximum. We calculated Q1, Q3, and the IQR in the previous notebook. The box plot gives us those same numbers as a picture.

The box spans from Q1 to Q3 (the IQR). The line inside the box is the median. The whiskers extend to the most extreme data points within 1.5 * IQR of the box. Points beyond the whiskers are plotted individually as **outliers**.

```
         outlier     whisker    Q1   median   Q3    whisker     outlier
    o       |----------[=========|==========]---------|
```

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(df["wait_to_treatment_min"].dropna(), vert=False)
plt.xlabel("Wait to treatment (minutes)")
plt.title("Box plot of wait to treatment")
plt.show()

The cluster of dots on the right are outliers: patients who waited far longer than the typical range. In an ED, each of those dots represents a real person. These are not noise to be filtered out. They are the patients the board should be asking about.

### Using seaborn for box plots

Seaborn's `boxplot` function produces the same chart with less code and better defaults for styling.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["total_time_in_ed_min"])
plt.xlabel("Total time in ED (minutes)")
plt.title("Box plot of total ED time")
plt.show()

---

## 4. Comparing distributions across groups

Box plots become especially powerful when we place them side by side. The explainer talked about how a multimodal distribution might signal distinct subgroups in the data. Grouping by triage category is one way to check whether the overall distribution is actually several different distributions stacked on top of each other.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="triage_category", y="wait_to_treatment_min")
plt.xlabel("Triage category")
plt.ylabel("Wait to treatment (minutes)")
plt.title("Wait to treatment by triage category")
plt.show()

Triage categories 1 (resuscitation) and 2 (emergency) have much shorter and tighter wait times. Categories 3 through 5 show longer waits and more spread. This is the expected pattern: the most urgent patients are seen first. If we had only looked at the overall histogram, this structure would have been invisible.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="triage_category", y="total_time_in_ed_min")
plt.xlabel("Triage category")
plt.ylabel("Total time in ED (minutes)")
plt.title("Total ED time by triage category")
plt.show()

### Comparing by arrival mode

Does how a patient arrives affect how long they wait? Ambulance patients may be triaged faster, while walk-ins may join a longer queue.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="arrival_mode", y="wait_to_treatment_min")
plt.xlabel("Arrival mode")
plt.ylabel("Wait to treatment (minutes)")
plt.title("Wait to treatment by arrival mode")
plt.show()

### Overlapping histograms

Another way to compare distributions is to overlay histograms with transparency. This works well when we want to see how the full shapes differ, not just the summary boxes.

In [ ]:
plt.figure(figsize=(10, 5))

for cat in [1, 2, 3, 4, 5]:
    subset = df[df["triage_category"] == cat]["wait_to_treatment_min"].dropna()
    plt.hist(subset, bins=30, alpha=0.4, label=f"Triage {cat}")

plt.xlabel("Wait to treatment (minutes)")
plt.ylabel("Number of patients")
plt.title("Wait to treatment by triage category")
plt.legend()
plt.show()

Triage 1 and 2 are concentrated on the left. Triage 3 to 5 spread further to the right. The overlapping histograms make the separation vivid in a way that summary statistics alone cannot.

---

## 5. Identifying skewness

We have seen skewness visually. Now let's confirm it numerically. **Skewness** measures the asymmetry of a distribution.

- **Right-skewed** (positive skew): the tail extends to the right. Mean > Median.
- **Left-skewed** (negative skew): the tail extends to the left. Mean < Median.
- **Symmetric**: tails are roughly equal. Mean ≈ Median.

Pandas provides `.skew()` to calculate this directly.

In [ ]:
cols = ["wait_to_triage_min", "wait_to_treatment_min", "total_time_in_ed_min"]

for c in cols:
    skew_val = df[c].skew()
    mean_val = df[c].mean()
    median_val = df[c].median()
    print(f"{c}:")
    print(f"  Skewness: {skew_val:.2f}  |  Mean: {mean_val:.1f}  |  Median: {median_val:.1f}")

A skewness value near 0 means roughly symmetric. Positive values indicate right skew. All three wait time columns are right-skewed, which confirms what the histograms showed. When we see the numbers and the pictures agreeing, we can report the finding with confidence.

---

## 6. Comparing distributions by shift

The ED does not operate in a vacuum. Staffing levels change between day and night shifts, and that could affect how long patients wait. Load the staff shifts data to see whether the staffing picture looks different by shift.

In [ ]:
staff = pd.read_csv("../data/ed_staff_shifts.csv")
staff.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=staff, x="shift", y="doctors_on_duty", ax=axes[0])
axes[0].set_title("Doctors on duty by shift")

sns.boxplot(data=staff, x="shift", y="nurses_on_duty", ax=axes[1])
axes[1].set_title("Nurses on duty by shift")

plt.tight_layout()
plt.show()

Night shifts typically have fewer staff. Whether this matters depends on whether patient arrivals also drop at night. That is a question about relationships between variables, and it is exactly the kind of thing we will investigate in notebook 3 when we bring all three datasets together.

---

## 7. Exercises

Time to practise. Complete each exercise in the code cell provided.

### Exercise 1: Histogram of wait to triage

Create a histogram of `wait_to_triage_min` using seaborn's `histplot` with a KDE overlay. Use 30 bins. Add axis labels and a title. In a comment, describe the shape of the distribution (symmetric, right-skewed, or left-skewed).

In [ ]:
# Your code here


### Exercise 2: Box plots by age group

Create a box plot showing `total_time_in_ed_min` for each `age_group`. Use seaborn. Which age group has the widest spread (largest IQR)?

In [ ]:
# Your code here


### Exercise 3: Side-by-side histograms by outcome

Create overlapping histograms of `total_time_in_ed_min` for each outcome (discharged, admitted, left without treatment, transferred). Use `alpha=0.4` for transparency. Add a legend. In a comment, describe any differences you observe.

In [ ]:
# Your code here


### Exercise 4: Skewness comparison

For each triage category (1 through 5), calculate the skewness of `wait_to_treatment_min`. Print the results. Which triage category has the most skewed wait times? Does this make sense given what the box plots showed?

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Built **histograms** with `plt.hist()` and `sns.histplot()` to see the shape of data
- Experimented with the `bins` parameter to find an appropriate level of detail
- Built **box plots** with `plt.boxplot()` and `sns.boxplot()` showing the five-number summary and outliers
- Compared distributions across triage categories, arrival modes, and shifts
- Identified **right skew** both visually and numerically with `.skew()`

We now have the two core skills of descriptive statistics: summarising data with numbers (notebook 1) and visualising its shape (this notebook). In the next notebook we will combine all three datasets and carry out a full exploratory data analysis, following the systematic EDA workflow from the explainer.